# All four models side by side

One table, three questions: does size buy copy fidelity, does it change dish-name perplexity, does it move
zero-shot BLEU. **Base is compared with base, large with large.**

In [ ]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
import csv
def table(name):
    p = ROOT / "results" / name
    return {r["model"]: r for r in csv.DictReader(p.open(encoding="utf-8"))} if p.exists() else {}

copy_t, ppl_t, bleu_t = table("copy_summary.csv"), table("ppl_summary.csv"), table("bleu_summary.csv")
fmt = "{:<14}{:>9}{:>9}{:>10}{:>10}{:>9}"
print(fmt.format("model", "params", "exact", "copyBLEU", "ppl_dish", "d_nll"))
for k in C.MODEL_ORDER:
    c, p = copy_t.get(k), ppl_t.get(k)
    if not c:
        print(fmt.format(k, "-", "-", "-", "-", "-"))
        continue
    print(fmt.format(k, C.MODELS[k]["size_class"], c["exact_match"], c["bleu"],
                     p["ppl_dish"] if p else "-", p["delta_nll"] if p else "-"))

## Zero-shot translation (one model per direction, matched size)

In [ ]:
print(f"{'model':<14}{'dir':<8}{'BLEU':>8}{'chrF':>8}{'dish_kept':>11}")
for k, r in bleu_t.items():
    print(f"{r['model']:<14}{r['direction']:<8}{r['bleu']:>8}{r['chrf']:>8}{r['dish_name_kept']:>11}")

## What the table does and does not license

- ✅ "On this machine, at this size, the copy prompt works/does not work."
- ✅ "Dish tokens are X nats harder than the rest of the sentence, with a paired CI."
- ❌ "ViT5 is better than Flan-T5" — different languages, different tokenisers, different test text.
- ❌ "Large is better" — the large models see the same data but with a different token budget; a difference
  in `exact_match` can come from decoding length alone (see `mean_hyp_words` in the copy table).